<a href="https://colab.research.google.com/github/Realbenjizo/python-functions-for-data-engineers/blob/main/Chapter_4_From_Individual_Functions_to_Composable_Pipelines.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np
from datetime import datetime
from typing import Dict, Optional, TypedDict

In [2]:
# Replace link with your link to downloaded files
transactions = "/content/drive/MyDrive/data_files/transactions.csv"

# Creating a DataFrame
transactions_df = pd.read_csv(transactions)
transactions_df['transaction_date']=pd.to_datetime(transactions_df['transaction_date'])

print(transactions_df)

   user_id  amount transaction_date     status
0      1.0  250.50       2025-01-03  completed
1      2.0    0.00       2025-02-03    pending
2      1.0  175.75       2025-01-03  completed
3      3.0   89.99       2025-03-03  completed
4      2.0  -10.00       2025-02-03     failed
5      4.0  320.00       2025-04-03  completed
6      NaN  150.00       2025-01-03  completed
7      1.0  200.00       2025-05-03  completed


In [3]:
user_lookup: Dict[int, str] = {
    1: "Alice Johnson",
    2: "Bob Smith",
    3: "Carol Williams",
    4: "David Brown"
}

## **Why composition matters**

In [4]:
from typing import Dict

# Step 1: Remove rows with null IDs
def remove_null_users(df: pd.DataFrame) -> pd.DataFrame:
    """Remove rows with missing user_id."""
    return df.dropna(subset=['user_id']).copy()

# Step 2: Map IDs to Names
def map_user_names(df: pd.DataFrame,
                   lookup: Dict[int, str]) -> pd.DataFrame:
    """Adds a 'user_name' column using lookup dictionary."""
    results_df = df.copy()
    results_df['user_name'] = results_df['user_id'].astype(int).map(lookup)
    return results_df

# Step 3: Filter for High-Value Transactions
def filter_high_value(df: pd.DataFrame,
                      threshold: float = 100.0) -> pd.DataFrame:
    """Returns only rows high value transactions."""
    return df[df['amount'] > threshold].copy()

## **Factory Functions for connectors**

In [5]:
import sqlite3
import os
from contextlib import contextmanager
from typing import Iterator, TextIO

# 1. Source connector
@contextmanager
def create_file_stream(file_path: str) -> Iterator[TextIO]:
    """Opens and yields a readable file stream, closing it automatically."""
    if not os.path.exists(file_path):
        raise FileNotFoundError(f"Source file not found at: {file_path}")
    file_obj = open(file_path, mode="r", encoding="utf-8")
    try:
        yield file_obj
    finally:
        file_obj.close()

# 2. Destination Connectors
def create_db_connection(db_path: str = ":memory:") -> sqlite3.Connection:
    """Create and return a new database connection."""
    return sqlite3.connect(db_path)

@contextmanager
def create_db_cursor(db_path: str = ":memory:") -> Iterator[sqlite3.Cursor]:
    """"Create a cursor and manage its database transaction and connection """
    conn = create_db_connection(db_path)
    try:
        yield conn.cursor()
        conn.commit()
    except Exception:
        conn.rollback()
        raise
    finally:
        conn.close()

## **ETL Functions that Use the Factory**

In [6]:
import pandas as pd

def load_transactions(resource_path: str) -> pd.DataFrame:
    """Extract: Load raw data from CSV."""
    with create_file_stream(resource_path) as file:
        raw_df = pd.read_csv(file)
    return raw_df

def save_to_database(df: pd.DataFrame, table_name: str,
                     db_path: str = "warehouse.db") -> None:
    """Load: Save DataFrame to database using connection factory."""
    with create_db_cursor(db_path) as cursor:
        df.to_sql(table_name, con=cursor.connection,
                  if_exists="replace", index=False)

### **For demonstration purpose only**

In [7]:
# Environment-aware factory pattern
def get_database_path(environment: str = "development") -> sqlite3.Connection:
    """Return a database connection configured for the target environment."""
    db_targets = {
        "test": ":memory:",
        "development": "dev_warehouse.db",
        "production": "/secure/prod/warehouse.db",
    }
    if environment not in db_targets:
        raise ValueError(f"Unsupported environment: {environment}")
    return sqlite3.connect(db_targets[environment])

## **Composing Functions into Pipelines**

### **1. Explict Sequential Compostion (FULL code)**

In [14]:
import sqlite3
import pandas as pd
import os
from contextlib import contextmanager
from typing import Iterator, TextIO

@contextmanager
def create_file_stream(file_path: str) -> Iterator[TextIO]:
    """Opens and yields a readable file stream, closing it automatically."""
    if not os.path.exists(file_path):
        raise FileNotFoundError(f"Source file not found at: {file_path}")
    file_obj = open(file_path, mode="r", encoding="utf-8")
    try:
        yield file_obj
    finally:
        file_obj.close()

def load_transactions(resource_path: str) -> pd.DataFrame:
    """Extract: Load raw data from CSV."""
    with create_file_stream(resource_path) as file:
        raw_df = pd.read_csv(file)
    return raw_df

# Step 1: Remove rows with null IDs
def remove_null_users(df: pd.DataFrame) -> pd.DataFrame:
    """Remove rows with missing user_id."""
    return df.dropna(subset=['user_id']).copy()

# Step 2: Map IDs to Names
def map_user_names(df: pd.DataFrame,
                   lookup: Dict[int, str]) -> pd.DataFrame:
    """Adds a 'user_name' column using lookup dictionary."""
    results_df = df.copy()
    results_df['user_name'] = results_df['user_id'].astype(int).map(lookup)
    return results_df

# Step 3: Filter for High-Value Transactions
def filter_high_value(df: pd.DataFrame,
                      threshold: float = 100.0) -> pd.DataFrame:
    """Returns only rows high value transactions."""
    return df[df['amount'] > threshold].copy()

@contextmanager
def create_db_cursor(db_path: str = ":memory:") -> Iterator[sqlite3.Cursor]:
    """"Create a cursor and manage its database transaction and connection """
    conn = create_db_connection(db_path)
    try:
        yield conn.cursor()
        conn.commit()
    except Exception:
        conn.rollback()
        raise
    finally:
        conn.close()

def save_to_database(df: pd.DataFrame, table_name: str,
                     db_path: str = "warehouse.db") -> None:
    """Load: Save DataFrame to database using connection factory."""
    with create_db_cursor(db_path) as cursor:
        df.to_sql(table_name, con=cursor.connection,
                  if_exists="replace", index=False)

# SEQUENTIAL composition
def run_complete_pipeline(lookup_table: Dict[int, str],
                          input_file: str,
                          db_path: str = "warehouse.db") -> pd.DataFrame:
    """Complete ETL pipeline with explicit steps."""

    # Extract (Impure I/O)
    raw_data = load_transactions(input_file)
    # Transform (Pure)
    clean_df = remove_null_users(raw_data)
    enriched_df = map_user_names(clean_df, lookup_table)
    final_df = filter_high_value(enriched_df, threshold=100.0)
    # Load (Impure I/O)
    save_to_database(final_df, table_name="high_value_transactions")

    return final_df

# Replace with link of your downloaded file
transactions = "/content/drive/MyDrive/data_files/transactions.csv"

# Execute the pipeline
if __name__ == "__main__":
    results = run_complete_pipeline(user_lookup, input_file=transactions)
    print(results)

   user_id  amount transaction_date     status      user_name
0      1.0  250.50        1 03 2025  completed  Alice Johnson
2      1.0  175.75        1 03 2025  completed  Alice Johnson
5      4.0  320.00        4 03 2025  completed    David Brown
7      1.0  200.00        5 03 2025  completed  Alice Johnson


### **2. Fluent Composition with .pipe() (Full code)**

In [15]:
@contextmanager
def create_file_stream(file_path: str) -> Iterator[TextIO]:
    """Opens and yields a readable file stream, closing it automatically."""
    if not os.path.exists(file_path):
        raise FileNotFoundError(f"Source file not found at: {file_path}")
    file_obj = open(file_path, mode="r", encoding="utf-8")
    try:
        yield file_obj
    finally:
        file_obj.close()

def load_transactions(resource_path: str) -> pd.DataFrame:
    """Extract: Load raw data from CSV."""
    with create_file_stream(resource_path) as file:
        raw_df = pd.read_csv(file)
    return raw_df

# Step 1: Remove rows with null IDs
def remove_null_users(df: pd.DataFrame) -> pd.DataFrame:
    """Remove rows with missing user_id."""
    return df.dropna(subset=['user_id']).copy()

# Step 2: Map IDs to Names
def map_user_names(df: pd.DataFrame,
                   lookup: Dict[int, str]) -> pd.DataFrame:
    """Adds a 'user_name' column using lookup dictionary."""
    results_df = df.copy()
    results_df['user_name'] = results_df['user_id'].astype(int).map(lookup)
    return results_df

# Step 3: Filter for High-Value Transactions
def filter_high_value(df: pd.DataFrame,
                      threshold: float = 100.0) -> pd.DataFrame:
    """Returns only rows high value transactions."""
    return df[df['amount'] > threshold].copy()

@contextmanager
def create_db_cursor(db_path: str = ":memory:") -> Iterator[sqlite3.Cursor]:
    """"Create a cursor and manage its database transaction and connection """
    conn = create_db_connection(db_path)
    try:
        yield conn.cursor()
        conn.commit()
    except Exception:
        conn.rollback()
        raise
    finally:
        conn.close()

def save_to_database(df: pd.DataFrame, table_name: str,
                     db_path: str = "warehouse.db") -> None:
    """Load: Save DataFrame to database using connection factory."""
    with create_db_cursor(db_path) as cursor:
        df.to_sql(table_name, con=cursor.connection,
                  if_exists="replace", index=False)


# Composition with pipe
def run_pipeline_with_pipe(lookup_table: Dict[int, str],
                           input_file: str,
                           db_path: str = "warehouse.db") -> pd.DataFrame:
    """Using pipe for the transformation chain."""
    # Extract (Impure I/O)
    raw_data = load_transactions(input_file)

    # Transform (Pure)
    processed_data = (raw_data
                      .pipe(remove_null_users)
                      .pipe(map_user_names, lookup_table)
                      .pipe(filter_high_value, threshold=100.0))

    # Load (Impure I/O)
    save_to_database(processed_data, table_name="high_value_transactions")

    return processed_data

# Replace with link of your downloaded file
transactions = "/content/drive/MyDrive/data_files/transactions.csv"

# Execute the pipeline
if __name__ == "__main__":
    results = run_pipeline_with_pipe(lookup_table=user_lookup, input_file=transactions)
    print(results)

   user_id  amount transaction_date     status      user_name
0      1.0  250.50        1 03 2025  completed  Alice Johnson
2      1.0  175.75        1 03 2025  completed  Alice Johnson
5      4.0  320.00        4 03 2025  completed    David Brown
7      1.0  200.00        5 03 2025  completed  Alice Johnson


### **3. Using functools.partial in pipelines**

In [10]:
from functools import partial

def filter_high_value(df: pd.DataFrame,
                      threshold: float) -> pd.DataFrame:
    """Returns only rows where amount exceeds the threshold."""
    return df[df['amount'] > threshold].copy()

# Using partial to set a constant parameter
high_value_filter = partial(filter_high_value, threshold=100.0)

## **Full code**

In [16]:
@contextmanager
def create_file_stream(file_path: str) -> Iterator[TextIO]:
    """Opens and yields a readable file stream, closing it automatically."""
    if not os.path.exists(file_path):
        raise FileNotFoundError(f"Source file not found at: {file_path}")
    file_obj = open(file_path, mode="r", encoding="utf-8")
    try:
        yield file_obj
    finally:
        file_obj.close()

def load_transactions(resource_path: str) -> pd.DataFrame:
    """Extract: Load raw data from CSV."""
    with create_file_stream(resource_path) as file:
        raw_df = pd.read_csv(file)
    return raw_df

# Step 1: Remove rows with null IDs
def remove_null_users(df: pd.DataFrame) -> pd.DataFrame:
    """Remove rows with missing user_id."""
    return df.dropna(subset=['user_id']).copy()

# Step 2: Map IDs to Names
def map_user_names(df: pd.DataFrame,
                   lookup: Dict[int, str]) -> pd.DataFrame:
    """Adds a 'user_name' column using lookup dictionary."""
    results_df = df.copy()
    results_df['user_name'] = results_df['user_id'].astype(int).map(lookup)
    return results_df

# Step 3: Filter for High-Value Transactions
def filter_high_value(df: pd.DataFrame,
                      threshold: float = 100.0) -> pd.DataFrame:
    """Returns only rows high value transactions."""
    return df[df['amount'] > threshold].copy()

@contextmanager
def create_db_cursor(db_path: str = ":memory:") -> Iterator[sqlite3.Cursor]:
    """"Create a cursor and manage its database transaction and connection """
    conn = create_db_connection(db_path)
    try:
        yield conn.cursor()
        conn.commit()
    except Exception:
        conn.rollback()
        raise
    finally:
        conn.close()

def save_to_database(df: pd.DataFrame, table_name: str,
                     db_path: str = "warehouse.db") -> None:
    """Load: Save DataFrame to database using connection factory."""
    with create_db_cursor(db_path) as cursor:
        df.to_sql(table_name, con=cursor.connection,
                  if_exists="replace", index=False)


def run_pipeline_with_pipe(lookup_table: Dict[int, str],
                           input_file: str,
                           db_path: str = "warehouse.db") -> pd.DataFrame:
    """Using pipe for the transformation chain."""
    # Extract (Impure I/O)
    raw_data = load_transactions(input_file)

    # Transform (Pure)
    processed_data = (raw_data
                      .pipe(remove_null_users)
                      .pipe(map_user_names, lookup_table)
                      .pipe(high_value_filter)) # configured function
    # Load (Impure I/O)
    save_to_database(processed_data, table_name="high_value_transactions")

    return processed_data

# File path (Replace with your downloaded file path)
transactions = "/content/drive/MyDrive/data_files/transactions.csv"

# Execute the pipeline
if __name__ == "__main__":
    results=run_pipeline_with_pipe(lookup_table=user_lookup, input_file=transactions)
    print(results)

   user_id  amount transaction_date     status      user_name
0      1.0  250.50        1 03 2025  completed  Alice Johnson
2      1.0  175.75        1 03 2025  completed  Alice Johnson
5      4.0  320.00        4 03 2025  completed    David Brown
7      1.0  200.00        5 03 2025  completed  Alice Johnson


### **Challenge Solution**

In [18]:
import pandas as pd

# 1. IMPURE FUNCTIONS (I/O Operations)
def load_data(file_path: str) -> pd.DataFrame:
    """Loads transaction data from a CSV file (Impure)."""
    with open(file_path, mode="r", encoding="utf-8") as file:
        return pd.read_csv(file)

# 2. PURE FUNCTIONS (Data Transformations)
def drop_nulls(df: pd.DataFrame) -> pd.DataFrame:
    """Removes rows with any null values (Pure)."""
    return df.dropna().copy()

def calculate_profit(df: pd.DataFrame) -> pd.DataFrame:
    """Adds a profit column calculated as amount - (amount * 0.25) (Pure)."""
    df_copy = df.copy()
    df_copy["profit"] = df_copy["amount"] - (df_copy["amount"] * 0.25)
    return df_copy

def filter_high_value(df: pd.DataFrame, threshold: float = 100.0) -> pd.DataFrame:
    """Returns only rows where profit ≥ threshold (Pure)."""
    df_copy = df.copy()
    return df_copy[df_copy["profit"] >= threshold].copy()

def run_pipeline(input_path: str,
                 threshold:float = 100.0
                 ):
    """
    Execute the transaction processing pipeline.
    """
    # Impure: Extract (I/O)
    input_data = load_data(input_path)

    # Pure functions
    cleaned_df = drop_nulls(input_data)
    profit_df = calculate_profit(cleaned_df)
    high_value_df = filter_high_value(profit_df, threshold)
    return high_value_df

# Replace with link to your dowloaded file
transactions = "/content/drive/MyDrive/data_files/transactions.csv"

# 3. PIPELINE ORCHESTRATION (Sequential Calls)
if __name__ == "__main__":
    results=run_pipeline(input_path=transactions)
    print(results)

   user_id  amount transaction_date     status    profit
0      1.0  250.50        1 03 2025  completed  187.8750
2      1.0  175.75        1 03 2025  completed  131.8125
5      4.0  320.00        4 03 2025  completed  240.0000
7      1.0  200.00        5 03 2025  completed  150.0000
